# 05 - Árvore de decisão ajustada

## Objetivo

Continuar a experiência da árvore do notebook 004, ajustando `max_depth` e `min_samples_leaf` numa pesquisa curta. Mantemos as mesmas variáveis, a divisão estratificada fixa e a preparação comum. O baseline 004 obteve Macro F1 **0,3628** na validação e pontuação Kaggle **0,35**; a regressão logística regularizada do 002 obteve **0,4125** na validação e **0,44** no Kaggle. A nova submissão só será criada se a árvore ajustada atingir pelo menos a referência local de 0,4125.

## Ambiente, dados e divisão fixa

Carregamos os mesmos CSV e reutilizamos `preparar_dados()` e `make_stratified_split()` do módulo comum. A divisão é feita com `test_size=0.2` e `random_state=42`, tal como nas experiências anteriores. A validação fica reservada para a avaliação; o pré-processador da pipeline aprende apenas ao ajustar em `X_train`.

In [ ]:
import sys
from pathlib import Path

import pandas as pd
import sklearn

candidate_roots = [Path.cwd(), *Path.cwd().resolve().parents]
repo_root = next(
    (p for p in candidate_roots if (p / 'notebooks').is_dir() and (p / 'src' / 'preparar_dados.py').is_file()),
    None,
)
if repo_root is None:
    raise FileNotFoundError('Não foi possível localizar a raiz do projeto com notebooks/ e src/preparar_dados.py.')

sys.path.insert(0, str(repo_root))
from src.preparar_dados import make_model_pipeline, make_stratified_split, preparar_dados

train_path = repo_root / 'notebooks' / 'dataset_train.csv'
test_path = repo_root / 'notebooks' / 'dataset_test_X.csv'
if not train_path.is_file() or not test_path.is_file():
    raise FileNotFoundError(f'CSV de treino/teste não encontrados em {repo_root / "notebooks"}')

print(f'Python: {sys.version}')
print(f'scikit-learn: {sklearn.__version__}')
print(f'Raiz do projeto: {repo_root}')

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)
prepared = preparar_dados(train_df, test_df)
X_train, X_val, y_train, y_val = make_stratified_split(
    prepared['X_train'], prepared['y_train'], test_size=0.2, random_state=42
)

assert len(X_train) == 8769 and len(X_val) == 2193, (len(X_train), len(X_val))
assert X_train.columns.equals(X_val.columns), 'As colunas de treino e validação não estão alinhadas.'
print(f'Treino: {X_train.shape}; validação: {X_val.shape}')
print(f'Classes no alvo: {sorted(y_train.unique().tolist())}')

## Comparação com os notebooks 004 e 002

Na divisão fixa, a árvore baseline sem limite do 004 obteve Macro F1 **0,3628** (treino **1,0000**, profundidade 29, 2515 folhas; Kaggle **0,35**). A regressão logística regularizada do 002 obteve Macro F1 **0,4125** na validação (Kaggle **0,44**). Primeiro selecionamos a configuração por CV apenas em `X_train`/`y_train`; depois avaliamos uma vez a melhor na validação fixa. As variáveis e a partição são as mesmas. A normalização numérica é desligada na árvore; logo, comparamos modelos, não isolamos um único fator causal.

## Teste curto de profundidade e tamanho mínimo das folhas

Comparamos as nove combinações de `max_depth=[6, 10, 15]` e `min_samples_leaf=[5, 15, 30]`, mantendo os restantes parâmetros da árvore baseline. A seleção usa Macro F1 em 3 folds estratificados, com `shuffle=True` e `random_state=42`, exclusivamente dentro da parcela de treino. Cada fold recebe uma pipeline nova, pelo que a imputação e a codificação são ajustadas apenas nesse fold. A validação fixa não participa na seleção. Avisos emitidos pelos estimadores permanecem visíveis.

In [ ]:
from sklearn.tree import DecisionTreeClassifier

from sklearn.model_selection import GridSearchCV, StratifiedKFold

tree_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
tree_search = GridSearchCV(
    estimator=make_model_pipeline(
        DecisionTreeClassifier(
            criterion='gini',
            splitter='best',
            min_samples_split=2,
            class_weight=None,
            random_state=42,
        ),
        normalize_numeric=False,
    ),
    param_grid={
        'model__max_depth': [6, 10, 15],
        'model__min_samples_leaf': [5, 15, 30],
    },
    scoring='f1_macro',
    cv=tree_cv,
    n_jobs=1,
    refit=True,
    return_train_score=True,
    error_score='raise',
)
tree_search.fit(X_train, y_train)

cv_results = pd.DataFrame(tree_search.cv_results_)
cv_table = cv_results[[
    'param_model__max_depth',
    'param_model__min_samples_leaf',
    'mean_train_score',
    'mean_test_score',
    'std_test_score',
    'rank_test_score',
    'split0_test_score',
    'split1_test_score',
    'split2_test_score',
]].rename(columns={
    'param_model__max_depth': 'max_depth',
    'param_model__min_samples_leaf': 'min_samples_leaf',
    'mean_train_score': 'treino_macro_f1_medio',
    'mean_test_score': 'cv_macro_f1_medio',
    'std_test_score': 'cv_macro_f1_desvio_padrao',
    'rank_test_score': 'posicao',
    'split0_test_score': 'fold_1',
    'split1_test_score': 'fold_2',
    'split2_test_score': 'fold_3',
}).sort_values(['posicao', 'max_depth', 'min_samples_leaf'])
display(cv_table.reset_index(drop=True))

best_params = {
    'max_depth': tree_search.best_params_['model__max_depth'],
    'min_samples_leaf': tree_search.best_params_['model__min_samples_leaf'],
}
best_cv_macro_f1 = tree_search.best_score_
print(f'Melhores parâmetros por CV: {best_params}')
print(f'Macro F1 CV média: {best_cv_macro_f1:.4f}')
print(f'Macro F1 treino CV média: {tree_search.cv_results_["mean_train_score"][tree_search.best_index_]:.4f}')

## Avaliação única da melhor combinação na validação fixa

A validação é usada agora, depois de concluída a seleção por CV. O melhor estimador já foi reajustado automaticamente em todo `X_train` pela pesquisa e apenas prevê `X_val`; não se volta a ajustar a preparação. Comparamos a Macro F1 com a árvore baseline (0,3628) e com a regressão logística 002 (0,4125).

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, f1_score

best_tree_pipe = tree_search.best_estimator_
best_val_pred = best_tree_pipe.predict(X_val)
best_train_pred = best_tree_pipe.predict(X_train)
best_train_macro_f1 = f1_score(
    y_train, best_train_pred, labels=list(range(16)), average='macro', zero_division=0
)
best_val_macro_f1 = f1_score(
    y_val, best_val_pred, labels=list(range(16)), average='macro', zero_division=0
)
best_fitted_tree = best_tree_pipe.named_steps['model']
baseline_macro_f1 = 0.3628
logistic_reference_macro_f1 = 0.4125

print(f'Parâmetros selecionados: {best_params}')
print(f'Macro F1 CV média: {best_cv_macro_f1:.4f}')
print(f'Macro F1 treino (melhor árvore): {best_train_macro_f1:.4f}')
print(f'Macro F1 validação (melhor árvore): {best_val_macro_f1:.4f}')
print(f'Diferença vs. árvore baseline (0.3628): {best_val_macro_f1 - baseline_macro_f1:+.4f}')
print(f'Diferença vs. regressão logística 002 (0.4125): {best_val_macro_f1 - logistic_reference_macro_f1:+.4f}')
print(f'Profundidade / folhas: {best_fitted_tree.get_depth()} / {best_fitted_tree.get_n_leaves()}')
print(classification_report(
    y_val,
    best_val_pred,
    labels=list(range(16)),
    zero_division=0,
    target_names=[str(i) for i in range(16)],
))
best_confusion = confusion_matrix(y_val, best_val_pred, labels=list(range(16)))
display(pd.DataFrame(best_confusion, index=range(16), columns=range(16)))

## Submissão condicionada ao resultado

Só geramos uma nova submissão se a Macro F1 de validação da árvore ajustada atingir pelo menos **0,4125**, a referência do 002. Caso contrário, não criamos outro CSV. Se passar o limiar, ajustamos uma pipeline nova em todo o treino. Os `RowId` vêm do exemplo oficial, se disponível, ou dos identificadores da submissão 001 já validada; nunca usamos previsões de exemplo.

In [ ]:
if best_val_macro_f1 >= logistic_reference_macro_f1:
    final_pipe = make_model_pipeline(
        DecisionTreeClassifier(
            criterion='gini',
            splitter='best',
            max_depth=best_params['max_depth'],
            min_samples_split=2,
            min_samples_leaf=best_params['min_samples_leaf'],
            class_weight=None,
            random_state=42,
        ),
        normalize_numeric=False,
    )
    final_pipe.fit(prepared['X_train'], prepared['y_train'])
    test_predictions = final_pipe.predict(prepared['X_test']).astype(int)

    example_path = repo_root / 'example_submission_10.csv'
    previous_path = repo_root / 'submissions' / 'submission_001_regressao_logistica.csv'
    if example_path.is_file():
        row_reference = pd.read_csv(example_path, usecols=['RowId'])
        reference_source = example_path
    elif previous_path.is_file():
        row_reference = pd.read_csv(previous_path, usecols=['RowId'])
        reference_source = previous_path
    else:
        raise FileNotFoundError('É necessário example_submission_10.csv ou a submissão 001 validada para preservar RowId.')

    assert len(test_predictions) == len(row_reference) == 3655
    submission = row_reference.copy()
    submission['AGE_RANGE'] = test_predictions
    assert list(submission.columns) == ['RowId', 'AGE_RANGE']
    assert submission['RowId'].equals(row_reference['RowId'])
    assert pd.api.types.is_integer_dtype(submission['AGE_RANGE'])
    assert submission['AGE_RANGE'].between(0, 15).all()

    submission_dir = repo_root / 'submissions'
    submission_dir.mkdir(exist_ok=True, parents=True)
    submission_path = submission_dir / 'submission_005_arvore_decisao_ajustada.csv'
    suffix = 1
    while submission_path.exists():
        suffix += 1
        submission_path = submission_dir / f'submission_005_arvore_decisao_ajustada_{suffix}.csv'
    submission.to_csv(submission_path, index=False)

    saved = pd.read_csv(submission_path)
    assert saved.shape == (3655, 2)
    assert list(saved.columns) == ['RowId', 'AGE_RANGE']
    assert saved['RowId'].equals(row_reference['RowId'])
    assert pd.api.types.is_integer_dtype(saved['AGE_RANGE'])
    assert saved['AGE_RANGE'].between(0, 15).all()
    print(f'Submissão guardada em {submission_path}')
    print(f'RowId copiado de {reference_source}')
    print(saved['AGE_RANGE'].value_counts().sort_index())
else:
    print(
        f'Sem nova submissão: Macro F1 de validação {best_val_macro_f1:.4f} '
        f'abaixo da referência 002 ({logistic_reference_macro_f1:.4f}). '
        'A próxima experiência recomendada é Random Forest.'
    )

## Registo e conclusão
- Data: 9/11
- Melhor combinação por CV: `max_depth=10`, `min_samples_leaf=15`; Macro F1 média CV **0,3846** (desvio padrão **0,0052**).
- Macro F1 de validação da árvore ajustada: **0,4037**; treino: **0,5355**.
- Diferença face ao baseline 004 (0,3628): **+0,0409**.
- Diferença face à regressão 002 (0,4125): **−0,0088**.
- Como 0,4037 fica abaixo de 0,4125, não foi criada submissão 005. Próximo passo: Random Forest.